<a href="https://colab.research.google.com/github/amyylxn/Ontario-Price-Forecasting/blob/main/price_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import glob
import pandas as pd

# Only plain year files (no _v suffix, and not the rolling "most recent" file)
files = sorted(glob.glob(f'{DATA_DIR}/ieso/PUB_PriceHOEPPredispOR_[0-9][0-9][0-9][0-9].csv'))
files = [f for f in files if 2016 <= int(f[-8:-4]) <= 2025]

frames = []
for f in files:
    year = f[-8:-4]
    d = pd.read_csv(f, skiprows=3, usecols=['Date', 'Hour', 'HOEP'])
    print(year, d.shape)
    frames.append(d)

price = pd.concat(frames, ignore_index=True)
price['Date'] = pd.to_datetime(price['Date'])
price['timestamp'] = price['Date'] + pd.to_timedelta(price['Hour'] - 1, unit='h')
price = price.sort_values('timestamp').reset_index(drop=True)

print(price.shape)
price.head()

2016 (8784, 3)
2017 (8760, 3)
2018 (8760, 3)
2019 (8760, 3)
2020 (8784, 3)
2021 (8760, 3)
2022 (8760, 3)
2023 (8760, 3)
2024 (8784, 3)
2025 (2880, 3)
(81792, 4)


,Date,Hour,HOEP,timestamp
0,2016-01-01,1,-1.09,2016-01-01 00:00:00
1,2016-01-01,2,-2.41,2016-01-01 01:00:00
2,2016-01-01,3,-4.23,2016-01-01 02:00:00
3,2016-01-01,4,-4.4,2016-01-01 03:00:00
4,2016-01-01,5,-4.44,2016-01-01 04:00:00


In [7]:
# Duplicates and missing values
print(price['timestamp'].duplicated().sum(), "duplicate timestamps")
print(price['HOEP'].isna().sum(), "missing HOEP values")

# Gaps: every step between rows should be exactly 1 hour
gaps = price['timestamp'].diff().dropna()
print(gaps.value_counts())

# Overall range
print(price['timestamp'].min(), "to", price['timestamp'].max())

0 duplicate timestamps
0 missing HOEP values
timestamp
0 days 01:00:00    81791
Name: count, dtype: int64
2016-01-01 00:00:00 to 2025-04-30 23:00:00


In [8]:
path = f'{DATA_DIR}/ieso/PUB_Demand_2022.csv'
with open(path) as f:
    for _ in range(8):
        print(f.readline().rstrip())

\\Hourly Demand Report,,,
\\Created at 2023-01-31 07:30:13,,,
\\For 2022,,,
Date,Hour,Market Demand,Ontario Demand
2022-01-01,1,15150,13543
2022-01-01,2,15344,13254
2022-01-01,3,14686,12683
2022-01-01,4,14400,12491


In [9]:
dem = pd.read_csv(path, skiprows=3)
print(dem.shape)

dem['Date'] = pd.to_datetime(dem['Date'])
dem['timestamp'] = dem['Date'] + pd.to_timedelta(dem['Hour'] - 1, unit='h')

dem.info()
print(dem[['Market Demand', 'Ontario Demand']].describe())
print(dem['timestamp'].duplicated().sum(), "duplicate timestamps")
print(dem['timestamp'].diff().dropna().value_counts())

(8760, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Date            8760 non-null   datetime64[ns]
 1   Hour            8760 non-null   int64         
 2   Market Demand   8760 non-null   int64         
 3   Ontario Demand  8760 non-null   int64         
 4   timestamp       8760 non-null   datetime64[ns]
dtypes: datetime64[ns](2), int64(3)
memory usage: 342.3 KB
       Market Demand  Ontario Demand
count    8760.000000     8760.000000
mean    17647.694977    15704.322489
std      2385.870197     2309.608268
min     11675.000000    10485.000000
25%     15853.750000    13997.000000
50%     17362.000000    15561.000000
75%     19313.250000    17414.250000
max     24702.000000    22607.000000
0 duplicate timestamps
timestamp
0 days 01:00:00    8759
Name: count, dtype: int64


In [11]:
files = sorted(glob.glob(f'{DATA_DIR}/ieso/PUB_Demand_[0-9][0-9][0-9][0-9].csv'))
files = [f for f in files if 2016 <= int(f[-8:-4]) <= 2025]

frames = []
for f in files:
    d = pd.read_csv(f, skiprows=3, usecols=['Date', 'Hour', 'Ontario Demand'])
    print(f[-8:-4], d.shape)
    frames.append(d)

demand = pd.concat(frames, ignore_index=True)
demand['Date'] = pd.to_datetime(demand['Date'])
demand['timestamp'] = demand['Date'] + pd.to_timedelta(demand['Hour'] - 1, unit='h')
demand = demand[['timestamp', 'Ontario Demand']].sort_values('timestamp')

# Join onto the price table; a left join keeps your price range (ends April 30, 2025)
data = price.merge(demand, on='timestamp', how='left')

print(data.shape)
print(data['Ontario Demand'].isna().sum(), "missing demand values")
data.head()

2016 (8784, 3)
2017 (8760, 3)
2018 (8760, 3)
2019 (8760, 3)
2020 (8784, 3)
2021 (8760, 3)
2022 (8760, 3)
2023 (8760, 3)
2024 (8784, 3)
2025 (8759, 3)
(81792, 5)
0 missing demand values


,Date,Hour,HOEP,timestamp,Ontario Demand
0,2016-01-01,1,-1.09,2016-01-01 00:00:00,13417
1,2016-01-01,2,-2.41,2016-01-01 01:00:00,12968
2,2016-01-01,3,-4.23,2016-01-01 02:00:00,12395
3,2016-01-01,4,-4.4,2016-01-01 03:00:00,12228
4,2016-01-01,5,-4.44,2016-01-01 04:00:00,12116
